# BiomedCLIP + Paired Text Fusion for Chest X-Ray Classification

### Journal Club 2026: Section 2 note notebook

This notebook sketches a downstream classifier built on the CLIP idea, but replaces a general CLIP model with **BiomedCLIP**:

- **Foundation model**: `microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224`
- **Image encoder**: biomedical Vision Transformer
- **Text encoder**: PubMedBERT
- **Downstream head**: a small MLP for three chest X-ray classes
- **Input idea**: each image is paired with a short clinical text prompt such as `chest x-ray of covid-19 pneumonia`
- **Fusion idea**: combine image and text embeddings, then classify the fused representation

The goal is not to build a clinical model. The goal is to demonstrate how multimodal foundation-model embeddings can be adapted for a focused medical-imaging classification task.


---
## 1. Concept

Classic CLIP learns two aligned embedding spaces:

1. An image encoder maps an image to an embedding.
2. A text encoder maps a caption or prompt to an embedding.
3. Contrastive learning pulls matched image-text pairs together and pushes mismatched pairs apart.

For a downstream classification task, there are at least two natural strategies:

- **Zero-shot CLIP**: compare an image embedding to class-prompt embeddings and choose the closest prompt.
- **Fusion classifier**: pair each image with a task-specific text prompt, concatenate or combine the two embeddings, and train a lightweight classifier.

This notebook uses the second strategy.


---
## 2. Install Dependencies

BiomedCLIP is loaded through OpenCLIP from the Hugging Face Hub. The first run downloads the model weights and tokenizer files.


In [ ]:
RUN_INSTALLS = True

if RUN_INSTALLS:
    import subprocess
    import sys

    packages = [
        "open_clip_torch==2.23.0",
        "transformers==4.35.2",
        "huggingface_hub",
        "pillow",
        "scikit-learn",
        "matplotlib",
        "seaborn",
        "tqdm",
    ]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])
else:
    print("Skipping installation. Set RUN_INSTALLS = True if imports fail.")


---
## 3. Imports and Reproducibility


In [ ]:
from pathlib import Path
import random

import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")


---
## 4. Load BiomedCLIP from Hugging Face

The official model card loads BiomedCLIP with OpenCLIP using the Hugging Face Hub model identifier. The model has already been pretrained on biomedical image-text pairs, so we use it as a foundation encoder rather than training image and text towers from scratch.


In [ ]:
from open_clip import create_model_from_pretrained, get_tokenizer

MODEL_ID = "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224"
CONTEXT_LENGTH = 256

biomedclip, preprocess = create_model_from_pretrained(MODEL_ID)
tokenizer = get_tokenizer(MODEL_ID)

biomedclip = biomedclip.to(device)
biomedclip.eval()

print("BiomedCLIP loaded.")
print(f"Image preprocessing pipeline:\n{preprocess}")


---
## 5. Define the Three Classes and Paired Text Prompts

Each image receives a paired text phrase based on its class. These are intentionally simple. Later, you can test richer report-style prompts.


In [ ]:
CLASS_NAMES = ["Normal", "Viral Pneumonia", "COVID-19"]
LABEL_TO_INDEX = {name: idx for idx, name in enumerate(CLASS_NAMES)}

CLASS_PROMPTS = {
    "Normal": "chest x-ray with no acute cardiopulmonary abnormality",
    "Viral Pneumonia": "chest x-ray of viral pneumonia with bilateral airspace opacities",
    "COVID-19": "chest x-ray of covid-19 pneumonia with peripheral bilateral opacities",
}

for label, prompt in CLASS_PROMPTS.items():
    print(f"{label:15s} -> {prompt}")


---
## 6. Dataset Location

This notebook expects the Kaggle COVID-19 Radiography Database used in the previous CLIP notebook:

`/content/COVID-19_Radiography_Dataset`

Expected class folders:

- `Normal/images`
- `Viral Pneumonia/images`
- `COVID/images`

If you already ran the previous notebook, keep `DATA_DIR` as-is. Otherwise, run your Kaggle download notebook first, or set `DATA_DIR` to the extracted dataset path on your machine.


In [ ]:
DATA_DIR = Path("/content/COVID-19_Radiography_Dataset")

FOLDER_TO_LABEL = {
    "Normal": "Normal",
    "Viral Pneumonia": "Viral Pneumonia",
    "COVID": "COVID-19",
}

MAX_PER_CLASS = 400
VAL_SIZE = 0.20
BATCH_SIZE = 16

print(f"DATA_DIR = {DATA_DIR}")


---
## 7. Build Image-Text Samples

Each sample contains:

1. image path
2. paired class prompt
3. class label index

For a training example from the COVID folder, the paired text is:

`chest x-ray of covid-19 pneumonia with peripheral bilateral opacities`


In [ ]:
def collect_samples(data_dir, folder_to_label, max_per_class=None):
    samples = []
    for folder_name, class_name in folder_to_label.items():
        image_dir = Path(data_dir) / folder_name / "images"
        if not image_dir.is_dir():
            raise FileNotFoundError(f"Missing expected folder: {image_dir}")

        paths = sorted(
            p for p in image_dir.iterdir()
            if p.suffix.lower() in {".png", ".jpg", ".jpeg"}
        )
        if max_per_class is not None:
            rng = random.Random(SEED + LABEL_TO_INDEX[class_name])
            paths = rng.sample(paths, min(max_per_class, len(paths)))

        prompt = CLASS_PROMPTS[class_name]
        label_idx = LABEL_TO_INDEX[class_name]
        samples.extend((str(path), prompt, label_idx, class_name) for path in paths)

    return samples

samples = collect_samples(DATA_DIR, FOLDER_TO_LABEL, max_per_class=MAX_PER_CLASS)
labels = [sample[2] for sample in samples]

train_samples, val_samples = train_test_split(
    samples,
    test_size=VAL_SIZE,
    random_state=SEED,
    stratify=labels,
)

print(f"Total samples: {len(samples):,}")
print(f"Train samples: {len(train_samples):,}")
print(f"Val samples:   {len(val_samples):,}")


---
## 8. Dataset and Collate Function

The `Dataset` loads and preprocesses images. The `collate_fn` tokenizes the paired text prompts for the batch.


In [ ]:
class ChestXRayTextDataset(Dataset):
    def __init__(self, samples, image_preprocess):
        self.samples = list(samples)
        self.image_preprocess = image_preprocess

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        image_path, prompt, label_idx, class_name = self.samples[idx]
        image = Image.open(image_path).convert("RGB")
        image = self.image_preprocess(image)
        return image, prompt, label_idx, class_name


def make_collate_fn(tokenizer, context_length=CONTEXT_LENGTH):
    def collate_fn(batch):
        images, prompts, labels, class_names = zip(*batch)
        images = torch.stack(images)
        texts = tokenizer(list(prompts), context_length=context_length)
        labels = torch.tensor(labels, dtype=torch.long)
        return images, texts, labels, list(prompts), list(class_names)
    return collate_fn

train_ds = ChestXRayTextDataset(train_samples, preprocess)
val_ds = ChestXRayTextDataset(val_samples, preprocess)

collate_fn = make_collate_fn(tokenizer)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, collate_fn=collate_fn)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, collate_fn=collate_fn)

batch = next(iter(train_loader))
print("Image batch:", batch[0].shape)
print("Text token batch:", batch[1].shape)
print("Label batch:", batch[2].shape)
print("Example prompt:", batch[3][0])


---
## 9. Inspect BiomedCLIP Embedding Size

We run one small batch through the frozen foundation model to determine the embedding dimension. For CLIP-style models, image and text embeddings should live in the same shared dimension.


In [ ]:
images, texts, labels, prompts, class_names = batch
images = images.to(device)
texts = texts.to(device)

with torch.no_grad():
    image_features = biomedclip.encode_image(images)
    text_features = biomedclip.encode_text(texts)
    image_features = F.normalize(image_features, dim=-1)
    text_features = F.normalize(text_features, dim=-1)

EMBED_DIM = image_features.shape[-1]
print("Image features:", image_features.shape)
print("Text features: ", text_features.shape)
print("Embedding dim: ", EMBED_DIM)


---
## 10. Fusion MLP Classifier

For each paired image-text example, we build a combined embedding:

`[image_embedding, text_embedding, image_embedding * text_embedding, |image_embedding - text_embedding|]`

This gives the MLP access to:

- image-only information
- text-prompt information
- multiplicative alignment between modalities
- absolute difference between modalities

By default, BiomedCLIP is frozen and only the MLP head is trained. Set `FREEZE_BIOMEDCLIP = False` for fine-tuning, but expect much higher memory use.


In [ ]:
class BiomedCLIPFusionMLP(nn.Module):
    def __init__(self, clip_model, embedding_dim, num_classes=3, hidden_dim=512, dropout=0.20, freeze_clip=True):
        super().__init__()
        self.clip_model = clip_model
        self.freeze_clip = freeze_clip

        if freeze_clip:
            for param in self.clip_model.parameters():
                param.requires_grad = False

        fusion_dim = embedding_dim * 4
        self.classifier = nn.Sequential(
            nn.Linear(fusion_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, num_classes),
        )

    def encode_pair(self, images, texts):
        context = torch.no_grad() if self.freeze_clip else torch.enable_grad()
        with context:
            image_features = self.clip_model.encode_image(images)
            text_features = self.clip_model.encode_text(texts)
            image_features = F.normalize(image_features, dim=-1)
            text_features = F.normalize(text_features, dim=-1)

        fused = torch.cat([
            image_features,
            text_features,
            image_features * text_features,
            torch.abs(image_features - text_features),
        ], dim=-1)
        return fused

    def forward(self, images, texts):
        fused = self.encode_pair(images, texts)
        return self.classifier(fused)

FREEZE_BIOMEDCLIP = True
fusion_model = BiomedCLIPFusionMLP(
    clip_model=biomedclip,
    embedding_dim=EMBED_DIM,
    num_classes=len(CLASS_NAMES),
    freeze_clip=FREEZE_BIOMEDCLIP,
).to(device)

trainable = sum(p.numel() for p in fusion_model.parameters() if p.requires_grad)
total = sum(p.numel() for p in fusion_model.parameters())
print(f"Trainable parameters: {trainable:,} / {total:,}")


---
## 11. Training Loop

This loop trains the MLP head on top of BiomedCLIP fused embeddings.

For a quick classroom demonstration, keep `MAX_EPOCHS` small and `MAX_PER_CLASS` modest. For a real experiment, use a held-out test set and repeated runs.


In [ ]:
MAX_EPOCHS = 5
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4

optimizer = torch.optim.AdamW(
    [p for p in fusion_model.parameters() if p.requires_grad],
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)
criterion = nn.CrossEntropyLoss()

history = {"train_loss": [], "val_loss": [], "val_acc": []}

for epoch in range(1, MAX_EPOCHS + 1):
    fusion_model.train()
    train_loss_sum = 0.0

    for images, texts, labels, _, _ in tqdm(train_loader, desc=f"Epoch {epoch} train"):
        images = images.to(device)
        texts = texts.to(device)
        labels = labels.to(device)

        logits = fusion_model(images, texts)
        loss = criterion(logits, labels)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        train_loss_sum += loss.item() * labels.size(0)

    train_loss = train_loss_sum / len(train_loader.dataset)

    fusion_model.eval()
    val_loss_sum = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, texts, labels, _, _ in tqdm(val_loader, desc=f"Epoch {epoch} val"):
            images = images.to(device)
            texts = texts.to(device)
            labels = labels.to(device)

            logits = fusion_model(images, texts)
            loss = criterion(logits, labels)
            preds = logits.argmax(dim=-1)

            val_loss_sum += loss.item() * labels.size(0)
            correct += (preds == labels).sum().item()
            total += labels.numel()

    val_loss = val_loss_sum / len(val_loader.dataset)
    val_acc = correct / max(total, 1)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(
        f"Epoch {epoch:02d}/{MAX_EPOCHS} | "
        f"train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | val_acc={val_acc:.3f}"
    )


---
## 12. Plot Training Curves


In [ ]:
fig, ax1 = plt.subplots(figsize=(7, 4))
ax1.plot(history["train_loss"], marker="o", label="Train loss")
ax1.plot(history["val_loss"], marker="s", label="Val loss")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.legend(loc="upper left")

ax2 = ax1.twinx()
ax2.plot(history["val_acc"], marker="^", color="seagreen", label="Val accuracy")
ax2.set_ylabel("Accuracy")
ax2.set_ylim(0, 1)
ax2.legend(loc="upper right")

plt.title("BiomedCLIP Fusion MLP Training")
plt.tight_layout()
plt.show()


---
## 13. Evaluate the Fusion Classifier


In [ ]:
fusion_model.eval()
all_true = []
all_pred = []

with torch.no_grad():
    for images, texts, labels, _, _ in val_loader:
        images = images.to(device)
        texts = texts.to(device)
        logits = fusion_model(images, texts)
        preds = logits.argmax(dim=-1).cpu().numpy()
        all_pred.extend(preds)
        all_true.extend(labels.numpy())

print(classification_report(all_true, all_pred, target_names=CLASS_NAMES, zero_division=0))

cm = confusion_matrix(all_true, all_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("BiomedCLIP Fusion MLP Confusion Matrix")
plt.tight_layout()
plt.show()


---
## 14. Compare with Zero-Shot BiomedCLIP

A useful baseline is zero-shot classification: encode the image once, encode one text prompt per class, and classify by cosine similarity.

This baseline uses no MLP training. If the MLP does not beat this baseline, the fusion head is probably not adding useful task-specific information yet.


In [ ]:
CLASS_TEXTS = [CLASS_PROMPTS[name] for name in CLASS_NAMES]
class_tokens = tokenizer(CLASS_TEXTS, context_length=CONTEXT_LENGTH).to(device)

biomedclip.eval()
with torch.no_grad():
    class_text_features = biomedclip.encode_text(class_tokens)
    class_text_features = F.normalize(class_text_features, dim=-1)

zero_shot_true = []
zero_shot_pred = []

with torch.no_grad():
    for images, _, labels, _, _ in val_loader:
        images = images.to(device)
        image_features = biomedclip.encode_image(images)
        image_features = F.normalize(image_features, dim=-1)
        similarities = image_features @ class_text_features.T
        preds = similarities.argmax(dim=-1).cpu().numpy()
        zero_shot_pred.extend(preds)
        zero_shot_true.extend(labels.numpy())

print(classification_report(zero_shot_true, zero_shot_pred, target_names=CLASS_NAMES, zero_division=0))


---
## 15. Save the MLP Head

When BiomedCLIP is frozen, the saved checkpoint only needs the MLP head plus class metadata. The foundation model can be reloaded from Hugging Face.


In [ ]:
SAVE_PATH = Path("biomedclip_chest_xray_fusion_mlp.pt")

torch.save({
    "model_id": MODEL_ID,
    "class_names": CLASS_NAMES,
    "class_prompts": CLASS_PROMPTS,
    "embedding_dim": EMBED_DIM,
    "freeze_biomedclip": FREEZE_BIOMEDCLIP,
    "classifier_state_dict": fusion_model.classifier.state_dict(),
}, SAVE_PATH)

print(f"Saved MLP head checkpoint to: {SAVE_PATH.resolve()}")


---
## 16. Discussion Questions

1. Does adding a paired text prompt improve performance over image-only embeddings?
2. Should the paired text be a class name, a report-style sentence, or a richer differential diagnosis prompt?
3. What information is duplicated between the image and text embeddings, and what information is complementary?
4. If BiomedCLIP is frozen, are we adapting enough for chest X-ray disease classification?
5. If BiomedCLIP is fine-tuned, how do we avoid overfitting on a relatively small teaching dataset?
6. How should this be evaluated before any clinical interpretation is considered?

Important: this notebook is for research education and method exploration only. It is not a clinical decision-support system.
